In [ ]:
import xlwings as xw
import numpy as np
import pandas as pd
from scipy.stats import triang
import matplotlib.pyplot as plt
import os


In [ ]:
# import util functions from external util file
%run util.py

Note that pdfs will be ignored in this extraction

In [ ]:
def extract_data_from_folders(root_dir):
    data_dict = {}

    app = xw.App(visible=False)
    
    # Traverse the directory tree
    for folder_name in os.listdir(root_dir):
        folder_path = os.path.join(root_dir, folder_name)
        
        # Check if it's a folder (skip if not a directory)
        if os.path.isdir(folder_path):
            # Extract the numeric ID from the folder name (assuming it starts with a number)
            folder_id = ''.join([ch for ch in folder_name if ch.isdigit()])  # Gets the digits from folder name
            
            # Initialize a list to store file details for this folder
            file_data = []
            
            # Iterate over files in the folder
            for file_name in os.listdir(folder_path):
                if (".xls" not in file_name) or (not file_name.startswith("MP")):
                    continue
                
                file_path = os.path.join(folder_path, file_name)

                wb = app.books.open(file_path)

                non_empty_count = 0

                # Loop through all sheets
                for sheet in wb.sheets:
                    # Get used range (range that includes all non-empty cells)
                    used_range = sheet.used_range
                    values = used_range.value
                
                    # Flatten values to a 1D list
                    if isinstance(values, list):
                        flat_values = [item for row in values for item in (row if isinstance(row, list) else [row])]
                    else:
                        flat_values = [values]
                
                    # Count non-empty (non-None and non-empty-string) values
                    non_empty_count += sum(1 for val in flat_values if val not in (None, ""))
                wb.close()

                print(file_name)
  
                # Extract the number after "MP" in the filename
                # EXPECTS FORMAT: MP###_NAME where ### is a number
                file_number = ''.join([ch for ch in file_name.split('_')[0][2:] if ch.isdigit()])
                    
                # Add the extracted data to the list
                file_data.append({
                    'file_number': file_number,
                    'file_non_empty_cells': non_empty_count
                })
            
            # Store the extracted data in the dictionary
            data_dict[folder_id] = file_data
    app.quit() 
    return data_dict

# Define the root directory where the folders are located
root_directory = './raw_data'

# Call the function and get the dictionary
result_dict = extract_data_from_folders(root_directory)

print(result_dict)


In [ ]:

 # Path to the data file 
excel_file = './methodology_data.xlsx' 

 # Name of the sheet containing data
sheet_name = 'data' 

num_header_rows = 1 # number of rows with metadata or headers
num_header_columns = 3 # number of columns with metadata or headers


# Connect to Excel
# THIS OPENS AND EDITS THE EXCEL FILE ON YOUR COMPUTER!
wb = xw.Book(excel_file)
sheet = wb.sheets[sheet_name]

# Determine the range of columns and results
data_range = sheet.range("A1").expand()  # Automatically detect data range
num_columns = data_range.columns.count  # Total number of columns
num_rows = data_range.rows.count  # Total number of columns

In [ ]:
row_monitoring_period_raw_file_non_empty_cells = get_row_by_variable("monitoring_period_raw_file_non_empty_cells", sheet, num_rows) 
if (row_monitoring_period_raw_file_non_empty_cells == -1)
    except("Row monitoring_period_raw_file_non_empty_cells doesn't exist. Please add it manually.")
row_numeric_id = get_row_by_variable("numeric_id", sheet, num_rows) 
row_monitoring_period = get_row_by_variable("monitoring_period", sheet, num_rows) 
row_data_missing = get_row_by_variable("data_missing", sheet, num_rows)

for i in range(num_header_columns + 1, num_columns):
    if (int(sheet.cells(row_data_missing, i).value) == 1):
        continue
    
    project_id = str(int(sheet.cells(row_numeric_id, i).value))
    monitoring_period = str(int(sheet.cells(row_monitoring_period, i).value))

    try:
        sheet.cells(row_monitoring_period_raw_file_non_empty_cells, i).value = next(item['file_non_empty_cells'] for item in result_dict[project_id] if item['file_number'] == monitoring_period)
    except:
        print(f'Skipping {project_id} in monitoring period {monitoring_period}')
        

In [ ]:
row_monitoring_period_raw_file_non_empty_cells = get_row_by_variable("monitoring_period_raw_file_non_empty_cells", sheet, num_rows) 
row_year = get_row_by_variable("year", sheet, num_rows) 
row_numeric_id = get_row_by_variable("numeric_id", sheet, num_rows) 
row_platform = get_row_by_variable("platform", sheet, num_rows) 

file_non_empty_cells = sheet.range((row_monitoring_period_raw_file_non_empty_cells, num_header_columns + 1), (row_monitoring_period_raw_file_non_empty_cells, num_columns)).value
year = sheet.range((row_year, num_header_columns + 1), (row_year, num_columns)).value
numeric_id = sheet.range((row_numeric_id, num_header_columns + 1), (row_numeric_id, num_columns)).value
platform = sheet.range((row_platform, num_header_columns + 1), (row_platform, num_columns)).value

In [ ]:
df = pd.DataFrame({
    'Year': year,
    'Number of non-empty cells': file_non_empty_cells,
    'Numeric id': numeric_id,
    'Platform': platform
})
# filter
df = df[
    df['Number of non-empty cells'].notna()
]

# Assign color based on platform (GS is red)
df['Color'] = df['Platform'].apply(lambda x: 'red' if x == 'GS' else 'blue')

# Group by 'Year' and calculate the average file size for each year
# average_number_of_non_empty_cells = df.groupby('Year')['Number of non-empty cells'].mean()

# Linear regression
x = df['Year']
y = df['Number of non-empty cells']
coeffs = np.polyfit(x, y, deg=1)  # Linear fit (degree 1)
regression_line = np.poly1d(coeffs)

# Plot regression line
x_vals = np.linspace(x.min(), x.max(), 100)
plt.plot(x_vals, regression_line(x_vals), color='black', linestyle='--', label='Regression line')


# Plot the result
#plt.plot(average_number_of_non_empty_cells.index, average_number_of_non_empty_cells.values, marker='o', linestyle='-', color='b')  # Line plot with markers
#plt.scatter(df['Year'], df['Number of non-empty cells'], c=df['Color'])
plt.xlabel('Year')
plt.ylabel('Average Number nof Non-empty Cells')
plt.title('Average Number nof Non-empty Cells Over Years')
plt.grid(True)

# Show the plot
plt.show()